# Spatially Masked Regression on AJILE12: surrogates, lagged model and full masking

Ostadsharif Memar & Dehghani (2026), arXiv:2606.11415, on DANDI:000055 version 0.220127.0436
(the only published version; see `reproduce_memar_2026.ipynb` for the header, the version choice
and the full data check). Claims tested here: surrogate-ieeg, surrogate-order-ieeg,
lagged-ieeg-no-gain and full-mask-above-zero-ieeg (which uses the mask sweep's m = 1.0 values; the
m = 1.0 run is repeated here with this notebook's generator). Settings and deviations as in
`reproduce_memar_2026.ipynb`; the last section shows how much they change the score on one
subject.

In [ ]:
import os, time, zlib, json, resource
import numpy as np
import pandas as pd
from scipy import stats
import smr

T0 = time.time()
PAPER = "memar-2026"
def seed(claim):
    return zlib.crc32(f"{PAPER}/{claim}".encode())

DECIM, MAX_EPOCH = 10, 200
SUBJECTS = [f"sub-{i:02d}" for i in range(1, 13)]
if os.environ.get("SMR_TEST"):                 # development only: two subjects
    SUBJECTS = ["sub-07", "sub-04"]
IDX = [int(s[-2:]) - 1 for s in SUBJECTS]
print("dataset", smr.DANDISET, smr.VERSION)

## Data check

The trials of every subject, preprocessed as in `reproduce_memar_2026.ipynb` (reading a only; its data check covers the metadata, duplicates and the clock).

In [ ]:
t = time.time()
DATA = {}
for s in SUBJECTS:
    S = smr.load_subject(s, readings=("a",))
    DATA[s] = smr.drop_uncoordinated(S)
    x = DATA[s]["a"]
    print(s, "sessions", len(S["sessions"]), "trials", x.shape[0], "electrodes", x.shape[1], "samples", x.shape[2],
          "dropped without coordinates", DATA[s]["n_dropped_el"], f"range {x.min():.2f} to {x.max():.2f}, SD {x.std():.3f}",
          f"({time.time() - t:.0f} s)", flush=True)
print("trials in all", sum(S["a"].shape[0] for S in DATA.values()))
print("peak memory MB", resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e3)

In [ ]:
def run_all(m, rng, reading="a", **kw):
    # one SMR condition on every subject, in order, from one generator
    out = {}
    for s in SUBJECTS:
        S = DATA[s]
        out[s] = smr.run_smr(S[reading].astype(np.float32), smr.neighbours(S["xyz"]), m, rng,
                             decim=DECIM, max_epoch=MAX_EPOCH, **kw)
    return out

def subj(res):
    return np.array([res[s]["dc"].mean() for s in SUBJECTS])

def msd(v):
    return f"{np.mean(v):.3f} ± {np.std(v):.3f}"

## Instantaneous model at m = 1.0

The baseline for the surrogate and lagged comparisons, and the m = 1.0 values for
full-mask-above-zero-ieeg: the common pipeline at m = 1.0, reading (a), with the generator of
lagged-ieeg-no-gain (the lagged run below continues from it).

In [ ]:
t = time.time()
rng_lag = np.random.default_rng(seed("lagged-ieeg-no-gain"))
INST = run_all(1.0, rng_lag, "a")
inst = subj(INST)
print(f"instantaneous m = 1.0: {msd(inst)} ({time.time() - t:.0f} s)")
print(dict(zip(SUBJECTS, inst.round(3))))

## surrogate-ieeg

> On iEEG surrogates, intra-subject DistCorr falls from 0.553 to 0.181 (phase shuffle), 0.232
> (IAAFT) and 0.201 (block shuffle).

Surrogates of every preprocessed trial and channel, then the common pipeline at m = 1.0 (one
generator for the claim, used for the surrogate draws and the fits). Reading (b) of the criterion
(end-matched segments before the phase shuffle) is not run: trimming each trial and channel to its
own best-matching segment gives series of different lengths, and neither the paper nor the code
says how they are put back into trials of one length.

In [ ]:
TAB_SUR = {"phase": (0.181, 0.029), "iaaft": (0.232, 0.060), "block": (0.201, 0.168)}
t = time.time()
rng_sur = np.random.default_rng(seed("surrogate-ieeg"))
SUR = {}
for kind in ("phase", "iaaft", "block"):
    res = {}
    for s in SUBJECTS:
        S = DATA[s]
        xs = smr.surrogate(S["a"], kind, rng_sur)
        res[s] = smr.run_smr(xs, smr.neighbours(S["xyz"]), 1.0, rng_sur, decim=DECIM, max_epoch=MAX_EPOCH)
        del xs
    SUR[kind] = subj(res)
    v = SUR[kind]
    exp, tol = TAB_SUR[kind]
    print(f"{kind}: {msd(v)}; within {exp} ± {tol}: {abs(v.mean() - exp) <= tol}; below the original "
          f"({inst.mean():.3f}): {v.mean() < inst.mean()}; subjects below their original: {int((v < inst).sum())} of 12 "
          f"({time.time() - t:.0f} s)", flush=True)
print(pd.DataFrame({"subject": SUBJECTS, "original": inst, **SUR}).round(3).to_string(index=False))
ok = all(abs(SUR[k].mean() - TAB_SUR[k][0]) <= TAB_SUR[k][1] and SUR[k].mean() < inst.mean() for k in SUR)
print("all three within tolerance and below the original:", ok)

**Result: reproduced.** The averages are 0.201 (phase), 0.184 (IAAFT) and 0.229 (block), each within its tolerance and below the original 0.602, in every subject.

## surrogate-order-ieeg

> In iEEG, phase shuffling degrades reconstruction most on average, and IAAFT surrogates retain
> somewhat more reconstructability.

In [ ]:
avgs = {k: round(float(v.mean()), 3) for k, v in SUR.items()}
print("averages:", avgs)
print("phase shuffle lowest of the three:", min(avgs, key=avgs.get) == "phase")
print("IAAFT above phase (reading a):", avgs["iaaft"] > avgs["phase"], "; order of Table II, phase < block < IAAFT (reading b):",
      avgs["phase"] < avgs["block"] < avgs["iaaft"])
print("subjects with phase lowest:", int(sum(SUR["phase"][i] < min(SUR["iaaft"][i], SUR["block"][i]) for i in range(len(SUBJECTS)))), "of 12")

**Result: not reproduced.** IAAFT (0.184), not phase shuffling (0.201), gives the lowest average, in all 12 subjects; IAAFT is not above phase (reading a) and the order of Table II (reading b) does not hold.

## full-mask-above-zero-ieeg

> With the whole local neighborhood masked, iEEG reconstruction remains above zero, so the target
> signal is not determined by its immediate neighborhood alone.

The criterion takes the m = 1.0 values of the mask sweep; they are recomputed here (the run above)
to compare them with the phase-shuffled surrogate in one notebook. `masking_coverage.ipynb` prints
the sweep's own m = 1.0 values.

In [ ]:
print(f"(a) m = 1.0 average {inst.mean():.3f} > 0: {inst.mean() > 0}; smallest electrode value "
      f"{min(INST[s]['dc'].min() for s in SUBJECTS):.3f}")
print(f"(b) m = 1.0 average {inst.mean():.3f} against the phase-shuffled average {SUR['phase'].mean():.3f}: "
      f"above {inst.mean() > SUR['phase'].mean()}; subjects above their own phase-shuffled value: "
      f"{int((inst > SUR['phase']).sum())} of 12")

**Result: reproduced (on the criterion's assumed reading b).** With the whole neighbourhood masked the average is 0.602, above zero (reading a) and above the phase-shuffled average 0.201 (reading b), in 12 of 12 subjects.

## lagged-ieeg-no-gain

> Adding lagged predictors (0, 20, 30, 50 and 60 ms) barely changes iEEG reconstruction (0.554 ±
> 0.067 against 0.553 ± 0.068).

In [ ]:
t = time.time()
print("lag offsets in samples:", smr.lag_samples())
LAG = run_all(1.0, rng_lag, "a", lagged=True)
lag = subj(LAG)
d = lag - inst
print(f"lagged {msd(lag)} against instantaneous {msd(inst)} ({time.time() - t:.0f} s)")
print(f"lagged average within 0.554 ± 0.067: {abs(lag.mean() - 0.554) <= 0.067}; "
      f"difference {lag.mean() - inst.mean():+.4f}, {100 * (lag.mean() - inst.mean()) / inst.mean():+.1f}% of the "
      f"instantaneous average; under 10%: {abs(lag.mean() - inst.mean()) < 0.1 * inst.mean()}")
print("per subject lagged minus instantaneous:", dict(zip(SUBJECTS, d.round(4))))
w = stats.wilcoxon(lag, inst)
print(f"context only: Wilcoxon signed-rank over subjects, statistic {w.statistic}, p = {w.pvalue:.4f}; "
      f"subjects with a gain {int((d > 0).sum())} of 12")

**Result: reproduced.** The lagged average is 0.603 ± 0.060 against 0.602 ± 0.058 for the instantaneous model, within 0.554 ± 0.067 and a difference of +0.3% of the instantaneous average; 7 of 12 subjects gain, Wilcoxon p = 0.42.

## How much the speed deviations change the score

On sub-07, the m = 1.0 model is trained closer to the paper's code (every sample, early stopping
with a patience of 200 epochs, up to 10000 epochs) in two parts, for time: (1) every 10th sample
with early stopping up to 3000 epochs, (2) every sample for 100 epochs, against this attempt's
setting (every 10th sample, 200 epochs). The generator is the fourth of intra-ieeg-mean (its seed plus 3).

In [ ]:
S = DATA["sub-07"]
nbr = smr.neighbours(S["xyz"])
t = time.time()
chk = {}
for label, decim, me in (("this attempt: every 10th sample, 200 epochs", 10, 200),
                         ("every 10th sample, up to 3000 epochs, patience 200", 10, 3000),
                         ("every sample, 100 epochs", 1, 100)):
    r = smr.run_smr(S["a"], nbr, 1.0, np.random.default_rng(seed("intra-ieeg-mean") + 3), decim=decim, max_epoch=me)
    chk[label] = r["dc"].mean()
    print(f"{label}: mean DistCorr {r['dc'].mean():.4f}, epochs run {r['n_ep']}, "
          f"median best epoch {int(np.median(r['best_ep']))} ({time.time() - t:.0f} s)", flush=True)
print("largest difference from this attempt's setting:", round(max(abs(v - chk[list(chk)[0]]) for v in chk.values()), 4))

## Summary

| Claim | Outcome |
|---|---|
| surrogate-ieeg | reproduced (0.201, 0.184, 0.229) |
| surrogate-order-ieeg | not reproduced (IAAFT lowest, not phase) |
| full-mask-above-zero-ieeg | reproduced, on the criterion's assumed reading (0.602 against 0.201) |
| lagged-ieeg-no-gain | reproduced (0.603 against 0.602) |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the
reasoning and the differences from the paper's method. The speed deviations change the sub-07
score by at most 0.0003. The end-matched reading of the phase surrogate was not run. The other
differences and the dataset findings are listed in the summary of `reproduce_memar_2026.ipynb`.


In [ ]:
print(f"notebook ran {(time.time() - T0) / 60:.1f} min; peak memory "
      f"{resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6:.1f} GB")